# 第43课 · STFT 原理 — 短时傅里叶变换：给信号加时间戳，时频分辨率 tradeoff

**目标**：理解 STFT 如何把 1D 音频信号变成 2D 时频矩阵，掌握 `win_len`、`hop` 参数对时频分辨率的影响，动手实现 `frame_signal()`。

🔗 **Aurora 连接**：`aurora.audio.stft.frame_signal()` 和 `aurora.audio.stft.stft()` 是本节的目标实现，路径位于 `src/aurora/audio/stft.py`。

← **上一课**　[L42 · FFT 图形化](L42_visual_fft.ipynb)

> 上节课学习了 **FFT 图形化**：蝴蝶图 + 纯音 / 和弦 / 噪声的频谱形态对比。  
> 本课将探讨 **STFT 原理**。

## 本课剧情：FFT 的盲点——"什么时候"出现什么频率？

你听一段钢琴演奏，C 大调开头，中间转成 G 调，结尾回到 C。全曲做一次 FFT，你能看到 C 和 G 的频率都存在——但哪个先哪个后，全看不出来。

这就是 FFT 的盲点：**它只知道"有什么频率"，不知道"什么时候"**。

STFT（短时傅里叶变换，Short-Time Fourier Transform）的解法极其直接——像翻幻灯片一样，把信号切成一帧一帧的短片段，每帧单独做 FFT：

```
STFT[m, k] = FFT( x[m·hop : m·hop + win_len] × window )[k]
```

- `m`：第 m 帧（时间坐标）
- `k`：第 k 个频率桶（频率坐标）
- `hop`：相邻两帧的步进（决定时间分辨率）

结果是一个 2D 矩阵：行是时间帧，列是频率桶。把它热力图化就是**声谱图（spectrogram）**——音频可视化的标准格式。

**时频权衡（Uncertainty Principle in practice）**：

| win_len（帧长）| Δf = sr/win_len | Δt = win_len/sr | 适合场景 |
|---|---|---|---|
| 128 点 | 62.5 Hz/bin | 16 ms/帧 | 时间精确（瞬态） |
| 512 点 | 15.6 Hz/bin | 64 ms/帧 | 频率精确（音高） |
| 2048 点 | 3.9 Hz/bin | 256 ms/帧 | 频率极精确（歌唱） |

帧长越大，频率越精细；帧长越小，时间越清晰——无法两全。

本节任务：实现 `frame_signal(x, win_len, hop)` — 把 1D 信号切成 `(n_frames, win_len)` 的 2D 矩阵。

In [ ]:
import numpy as np
from aurora.audio.io import sine

## 概念 1：STFT 定义

```
STFT[m, k] = FFT( x[m*hop : m*hop+win_len] * window )[k]
```

- `m`：帧索引（时间轴）
- `k`：频率 bin 索引（频率轴）
- `window`：加权函数（Hann 等），压制帧边缘的频谱泄漏

输入是长度为 `N` 的 1D 信号，输出是 `(n_frames, win_len//2+1)` 的复数矩阵。取模得幅度谱，取角度得相位谱（phase spectrum）。

In [ ]:
# 演示：对单帧手动做 FFT，验证和 np.fft.rfft 的关系
x_demo = sine(freq=440, duration=0.01, sample_rate=8000)  # 80 个采样点
win_len = 32
window = np.hanning(win_len)

frame = x_demo[:win_len] * window
spectrum = np.fft.rfft(frame)  # 参考值，我们后面会手写 FFT

print(f'帧长: {win_len}, 频率 bins: {len(spectrum)}')
print(f'幅度谱前5个 bin: {np.abs(spectrum[:5]).round(4)}')

## 概念 2：时频权衡

```
频率分辨率（frequency resolution） = sr / win_len   （Hz/bin）
时间分辨率（time resolution） = hop / sr        （秒/帧）
```

- `win_len` 越大 → 每帧采样点（sampling point）多 → 频率分辨率高，但时间定位模糊
- `hop` 越小 → 帧与帧之间步长短 → 时间分辨率高，但计算量增加
- 两者存在根本性的权衡（不确定性原理在离散信号上的体现）

典型设置：`sr=22050`，`win_len=2048`（≈93ms），`hop=512`（≈23ms，75% 重叠）。

In [ ]:
# 演示：不同 win_len 下的频率/时间分辨率
sr = 8000
for win_len in [128, 512, 2048]:
    freq_res = sr / win_len
    for hop in [win_len//4, win_len//2]:
        time_res_ms = hop / sr * 1000
        print(f'win={win_len:5d}  hop={hop:5d}  '
              f'频率分辨率={freq_res:.1f}Hz  时间分辨率={time_res_ms:.1f}ms')

## 概念 3：帧数计算与补零（zero padding）

```
n_frames = 1 + (len(x) - win_len) // hop
```

这个公式只计算"完整帧"数量。最后一帧不足 `win_len` 时直接丢弃（center=False 模式）。如果需要覆盖全段信号，可在末尾补零到 `(n_frames-1)*hop + win_len`。

例：`len(x)=10`，`win_len=4`，`hop=2` → `n_frames = 1 + (10-4)//2 = 4`，帧起点为 `0, 2, 4, 6`。

In [ ]:
# 演示：帧数公式
for n, w, h in [(10,4,2),(100,32,16),(22050,2048,512)]:
    n_frames = 1 + (n - w) // h
    print(f'len={n:6d}  win={w:5d}  hop={h:5d}  → n_frames={n_frames}')

## 1. ✏️ 实现 `frame_signal(x, win_len, hop)`

**核心公式**：第 m 帧 = `x[m*hop : m*hop + win_len]`

**三步实现路线**：

| 步骤 | 代码思路 | 说明 |
|---|---|---|
| 1 | `n_frames = 1 + (len(x) - win_len) // hop` | 帧数公式（能取完整帧的数量） |
| 2 | `frames = np.zeros((n_frames, win_len))` | 预分配输出矩阵 |
| 3 | `frames[m] = x[m*hop : m*hop+win_len]` | 逐帧切片（循环 m=0..n_frames-1） |

**验收标准**：
- `frame_signal(np.arange(10), win_len=4, hop=2)` → shape `(4, 4)`
- `frames[0] = [0,1,2,3]`，`frames[1] = [2,3,4,5]`（步进 hop=2）
- 所有帧长度相同（余下不足 win_len 的尾部不取）

In [ ]:
def frame_signal(x: np.ndarray, win_len: int, hop: int) -> np.ndarray:
    """把 1D 信号切成重叠帧，返回 shape (n_frames, win_len)。"""
    # ✏️ TODO: 第1步 — 计算 n_frames
    raise NotImplementedError("TODO: 计算帧数 = (len(x) - n_fft) // hop_length + 1")

    # ✏️ TODO: 第2步 — 用 sliding_window_view 或循环构建 2D 帧矩阵
    raise NotImplementedError("TODO: 用 np.lib.stride_tricks.as_strided 或循环分帧")

    # ✏️ TODO: 第3步 — 返回 shape (n_frames, win_len) 的数组
    return frames


In [ ]:
# 检查 frame_signal
result = frame_signal(np.arange(10), 4, 2)
if result is None or result is ...:
    print('⬜ frame_signal 未实现')
else:
    assert result.shape == (4, 4), f'shape 错误: {result.shape}'
    assert list(result[0]) == [0, 1, 2, 3], f'第0帧错误: {result[0]}'
    assert list(result[1]) == [2, 3, 4, 5], f'第1帧错误: {result[1]}'
    assert list(result[3]) == [6, 7, 8, 9], f'第3帧错误: {result[3]}'
    print('✅ frame_signal 验证通过：shape', result.shape)
    print('帧矩阵：\n', result)

## 参数实验：win_len 对时间分辨率的影响

用同一段 0.5 秒的语音（sr=8000，N=4000 个采样点），分别设 `win_len=128`（16ms）和 `win_len=2048`（256ms），`hop` 都取 `win_len//4`。

**预期现象**：
- `win_len=128` → 帧数多（约 **122 帧**，公式：1+(4000-128)//32=122），时间分辨率高，但每帧频率 bin 少（**65 个**，即 win_len//2+1）
- `win_len=2048` → 帧数少（约 **4 帧**，公式：1+(4000-2048)//512=4），频率 bin 多（**1025 个**，即 win_len//2+1），但时间粒度粗
- 帧数之比（N=4000 小信号）≈ 122/4 ≈ **30.5 倍**；大信号（N≫win_len）时渐近 hop 之比 512/32 = **16 倍**

> ⚠️ 注：N=4000 仅是演示用小信号，2048 窗在 4000 点信号上只能完整切出 4 帧，导致比例远大于渐近值 16。大信号（如 sr=8000 × 10 秒 = 80000 点）时渐近规律才清晰。

In [ ]:
# 守卫：若 frame_signal 尚未实现，跳过本 cell 避免崩溃
_guard = frame_signal(np.arange(10), 4, 2)
if _guard is None or _guard is ...:
    print('⬜ 请先完成 frame_signal 练习，再运行本 cell')
    raise SystemExit(0)

# 参数实验：win_len 对帧数（时间分辨率）的影响
sr = 8000
duration = 0.5
x = sine(freq=440, duration=duration, sample_rate=sr)

for win_len in [128, 2048]:
    hop = win_len // 4
    frames = frame_signal(x, win_len, hop)
    freq_bins = win_len // 2 + 1
    time_res_ms = hop / sr * 1000
    freq_res_hz = sr / win_len
    print(f'win_len={win_len:5d}  hop={hop:5d}  '
          f'帧数={frames.shape[0]:4d}  '
          f'频率bins={freq_bins:5d}  '
          f'时间分辨率={time_res_ms:.1f}ms  '
          f'频率分辨率={freq_res_hz:.1f}Hz')

print()
f128 = frame_signal(x, 128, 32)
f2048 = frame_signal(x, 2048, 512)
print(f'帧数之比（128 vs 2048）: {f128.shape[0]} / {f2048.shape[0]} = {f128.shape[0]/f2048.shape[0]:.1f}x')

## 本课收束

`frame_signal(x, win_len, hop)` 把 1D 信号切成 shape `(n_frames, win_len)` 的 2D 矩阵，是 STFT 管道的第一步。`win_len` 和 `hop` 共同决定时频权衡：大窗口换频率精度，小 hop 换时间精度，两者无法同时最优。这个函数将直接嵌入 `aurora.audio.stft.stft()`，对每行乘以 Hann 窗再做 FFT，输出完整的复数时频矩阵。下一课（L44）将实现完整的 `stft()` 函数，并可视化真实音频的声谱图。

## ✏️ 白板挑战：STFT 手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`x` 长 100，`win_len=32`，`hop=16`，共有多少帧？  
（公式：`n_frames = 1 + (100-32)//16 = ?`）

**问 2**：`frame_signal(np.arange(10), 4, 2)` 应返回什么矩阵？  
（4 行 4 列，列出每行的值）

**问 3**：`sr=8000`，`win_len=512`：
- 频率分辨率 Δf = sr/win_len = ? Hz/bin
- 时间分辨率 Δt = win_len/sr = ? ms/帧

**问 4**：为什么 STFT 帧通常用 50% 重叠（`hop = win_len/2`）而不是 0 重叠？  
（提示：与上节 L41 Hann 窗 COLA 条件有关）

**问 5**：声谱图（spectrogram）的横轴/纵轴/颜色各表示什么？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：帧数公式
n, w, h = 100, 32, 16
n_frames_calc = 1 + (n - w) // h
assert n_frames_calc == 6
print(f"Q1 ✅  n_frames = 1 + (100-32)//16 = {n_frames_calc}")

# 问2：frame_signal(arange(10), 4, 2) 手算
x10 = np.arange(10, dtype=float)
expected = np.array([
    [0, 1, 2, 3],
    [2, 3, 4, 5],
    [4, 5, 6, 7],
    [6, 7, 8, 9],
], dtype=float)
try:
    result = frame_signal(x10, 4, 2)
    assert result is not None and result.shape == (4, 4)
    assert np.allclose(result, expected, atol=1e-12)
    print(f"Q2 ✅  frame_signal(arange(10),4,2) shape={result.shape}, frames[0]={result[0]}, frames[1]={result[1]}")
except NotImplementedError:
    print(f"Q2 ✅  手算结果：frames[0]={expected[0]}, frames[1]={expected[1]}（frame_signal 待实现）")

# 问3：sr=8000, win_len=512 时频分辨率
sr, win_len = 8000, 512
delta_f = sr / win_len
delta_t_ms = win_len / sr * 1000
assert delta_f == 15.625
assert abs(delta_t_ms - 64.0) < 0.001
print(f"Q3 ✅  Δf={delta_f} Hz/bin，Δt={delta_t_ms:.1f} ms/帧")

# 问4：50% 重叠 COLA 验证（N=8 Hann 窗）
N4 = 8
w4 = 0.5 * (1 - np.cos(2 * np.pi * np.arange(N4) / N4))
hop4 = N4 // 2
ola = w4[:hop4] + w4[hop4:]
assert np.std(ola) < 0.05 * np.mean(ola)
print(f"Q4 ✅  Hann窗50%重叠满足COLA：OLA均值={np.mean(ola):.4f}，标准差={np.std(ola):.6f}")

# 问5：声谱图轴含义（通过断言验证概念）
# 横轴=时间（帧m），纵轴=频率（bin k），颜色=|STFT[m,k]|或dB
assert True  # 概念问题，此处用 fft 验证频谱形状
sr5, T5 = 8000, 0.032
t5 = np.linspace(0, T5, int(sr5 * T5), endpoint=False)
x5 = np.sin(2 * np.pi * 440 * t5)
X5 = np.fft.rfft(x5 * np.hanning(len(x5)))
assert np.argmax(np.abs(X5)) > 0  # 440Hz峰不在DC
print(f"Q5 ✅  声谱图：横轴=时间帧，纵轴=频率bin，颜色=|STFT[m,k]|（通常取dB）")
print("\n🎉 STFT 白板挑战通过！n_frames=1+(N-win)//hop，Δf=sr/win，Δt=win/sr。")

In [ ]:
# ✏️ 本课自评
l43_review = {
    "stft_formula":            None,  # 记住 STFT[m,k]=FFT(x[m·hop:m·hop+win]×window)[k]？True/False
    "frame_signal_impl":       None,  # frame_signal 实现并通过 shape 断言？True/False
    "time_freq_tradeoff":      None,  # 理解大窗→高频率分辨率/低时间分辨率的权衡？True/False
    "n_frames_formula":        None,  # 记住 n_frames=1+(N-win_len)//hop？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l43_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l43_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L43 全部通关！进入 L44：亲手写 STFT')

---

→ **下一课**　[L44 · 亲手写 STFT](L44_stft_implement.ipynb)

> 下节课将学习 **亲手写 STFT**：分帧 + 加窗 + FFT，与 aurora.audio.stft 对齐验证。